In [2]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout, Bidirectional
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.callbacks import EarlyStopping

from googletrans import Translator

In [1]:
pip install googletrans==4.0.0-rc1

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.1/55.1 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 133.4/133.4 kB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.6/42.6 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.8/58.8 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 8.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 50.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.6/53.6 kB 5.3 MB/s eta 0:00:00
  Created wheel for googletrans: filename=googletrans-4.0.0rc1-py3-none-any.whl size=17396 sha256=5a10328bc46cd54792c56c631c268fbb2da4b1a876a364d5955a985bb0739def
  Stored in directory: /root/.cache/pip/wheels/95/0f/04/b17a72024b56a60e499ce1a6313d283ed5ba332407155bee03
Successfully built googletrans
  Attempting uninstall: hyperframe
    Found existing installation: hyperframe 6.1.0
    Uninstalling hyperfra

In [6]:
data =pd.read_csv("Language Detection.csv")
data

,Text,Language
0,"Nature, in the broadest sense, is the natural...",English
1,"""Nature"" can refer to the phenomena of the phy...",English
2,"The study of nature is a large, if not the onl...",English
3,"Although humans are part of nature, human acti...",English
4,[1] The word nature is borrowed from the Old F...,English
...,...,...
10332,ನಿಮ್ಮ ತಪ್ಪು ಏನು ಬಂದಿದೆಯೆಂದರೆ ಆ ದಿನದಿಂದ ನಿಮಗೆ ಒ...,Kannada
10333,ನಾರ್ಸಿಸಾ ತಾನು ಮೊದಲಿಗೆ ಹೆಣಗಾಡುತ್ತಿದ್ದ ಮಾರ್ಗಗಳನ್...,Kannada
10334,ಹೇಗೆ ' ನಾರ್ಸಿಸಿಸಮ್ ಈಗ ಮರಿಯನ್ ಅವರಿಗೆ ಸಂಭವಿಸಿದ ಎ...,Kannada
10335,ಅವಳು ಈಗ ಹೆಚ್ಚು ಚಿನ್ನದ ಬ್ರೆಡ್ ಬಯಸುವುದಿಲ್ಲ ಎಂದು ...,Kannada


In [7]:
print(data.head())

                                                Text Language
0   Nature, in the broadest sense, is the natural...  English
1  "Nature" can refer to the phenomena of the phy...  English
2  The study of nature is a large, if not the onl...  English
3  Although humans are part of nature, human acti...  English
4  [1] The word nature is borrowed from the Old F...  English


In [8]:
data = data.dropna()

In [9]:
texts = data["Text"].astype(str)
texts

,Text
0,"Nature, in the broadest sense, is the natural..."
1,"""Nature"" can refer to the phenomena of the phy..."
2,"The study of nature is a large, if not the onl..."
3,"Although humans are part of nature, human acti..."
4,[1] The word nature is borrowed from the Old F...
...,...
10332,ನಿಮ್ಮ ತಪ್ಪು ಏನು ಬಂದಿದೆಯೆಂದರೆ ಆ ದಿನದಿಂದ ನಿಮಗೆ ಒ...
10333,ನಾರ್ಸಿಸಾ ತಾನು ಮೊದಲಿಗೆ ಹೆಣಗಾಡುತ್ತಿದ್ದ ಮಾರ್ಗಗಳನ್...
10334,ಹೇಗೆ ' ನಾರ್ಸಿಸಿಸಮ್ ಈಗ ಮರಿಯನ್ ಅವರಿಗೆ ಸಂಭವಿಸಿದ ಎ...
10335,ಅವಳು ಈಗ ಹೆಚ್ಚು ಚಿನ್ನದ ಬ್ರೆಡ್ ಬಯಸುವುದಿಲ್ಲ ಎಂದು ...


In [11]:
languages = data["Language"]
languages

,Language
0,English
1,English
2,English
3,English
4,English
...,...
10332,Kannada
10333,Kannada
10334,Kannada
10335,Kannada


In [13]:
encoder = LabelEncoder()
labels = encoder.fit_transform(languages)
labels

array([3, 3, 3, ..., 9, 9, 9])

In [14]:
tokenizer = Tokenizer(
    char_level=True,
    lower=True,
    oov_token="<OOV>"
)

tokenizer.fit_on_texts(texts)

In [15]:
sequences = tokenizer.texts_to_sequences(texts)
sequences

[[2,
  6,
  4,
  8,
  13,
  9,
  3,
  19,
  2,
  5,
  6,
  2,
  8,
  18,
  3,
  2,
  24,
  9,
  7,
  4,
  11,
  3,
  10,
  8,
  2,
  10,
  3,
  6,
  10,
  3,
  19,
  2,
  5,
  10,
  2,
  8,
  18,
  3,
  2,
  6,
  4,
  8,
  13,
  9,
  4,
  12,
  19,
  2,
  16,
  18,
  29,
  10,
  5,
  14,
  4,
  12,
  19,
  2,
  15,
  4,
  8,
  3,
  9,
  5,
  4,
  12,
  2,
  28,
  7,
  9,
  12,
  11,
  2,
  7,
  9,
  2,
  13,
  6,
  5,
  20,
  3,
  9,
  10,
  3,
  21],
 [138,
  6,
  4,
  8,
  13,
  9,
  3,
  138,
  2,
  14,
  4,
  6,
  2,
  9,
  3,
  23,
  3,
  9,
  2,
  8,
  7,
  2,
  8,
  18,
  3,
  2,
  16,
  18,
  3,
  6,
  7,
  15,
  3,
  6,
  4,
  2,
  7,
  23,
  2,
  8,
  18,
  3,
  2,
  16,
  18,
  29,
  10,
  5,
  14,
  4,
  12,
  2,
  28,
  7,
  9,
  12,
  11,
  19,
  2,
  4,
  6,
  11,
  2,
  4,
  12,
  10,
  7,
  2,
  8,
  7,
  2,
  12,
  5,
  23,
  3,
  2,
  5,
  6,
  2,
  17,
  3,
  6,
  3,
  9,
  4,
  12,
  21],
 [8,
  18,
  3,
  2,
  10,
  8,
  13,
  11,
  29,
  2,
  7,
  23,
  2,
  6,
 

In [16]:
max_len = 200

X = pad_sequences(
    sequences,
    maxlen=max_len,
    padding='post'
)
X

array([[  2,   6,   4, ...,   0,   0,   0],
       [138,   6,   4, ...,   0,   0,   0],
       [  8,  18,   3, ...,   0,   0,   0],
       ...,
       [232, 245, 165, ...,   0,   0,   0],
       [256, 146, 212, ...,   0,   0,   0],
       [307, 133, 132, ...,   0,   0,   0]], dtype=int32)

In [17]:
y = np.array(labels)
y

array([3, 3, 3, ..., 9, 9, 9])

In [18]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [19]:
vocab_size = len(tokenizer.word_index) + 1

In [20]:
model = Sequential()

model.add(
    Embedding(
        input_dim=vocab_size,
        output_dim=128,
        input_length=max_len
    )
)

model.add(
    Bidirectional(
        LSTM(
            128,
            dropout=0.3,
            recurrent_dropout=0.3
        )
    )
)

model.add(Dropout(0.3))

model.add(
    Dense(
        128,
        activation='relu'
    )
)

model.add(Dropout(0.3))

model.add(
    Dense(
        len(set(labels)),
        activation='softmax'
    )
)

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [21]:
model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

In [22]:
early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=3,
    restore_best_weights=True
)

In [23]:
model.fit(
    X_train,
    y_train,
    epochs=15,
    batch_size=64,
    validation_data=(X_test, y_test),
    callbacks=[early_stop]
)

Epoch 1/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 187s 1s/step - accuracy: 0.3035 - loss: 2.0535 - val_accuracy: 0.4884 - val_loss: 1.4604
Epoch 2/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 169s 1s/step - accuracy: 0.5207 - loss: 1.3345 - val_accuracy: 0.6214 - val_loss: 0.9917
Epoch 3/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 166s 1s/step - accuracy: 0.5876 - loss: 1.0895 - val_accuracy: 0.6518 - val_loss: 0.8489
Epoch 4/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 205s 1s/step - accuracy: 0.6319 - loss: 0.9437 - val_accuracy: 0.6591 - val_loss: 0.8436
Epoch 5/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 198s 1s/step - accuracy: 0.6535 - loss: 0.8818 - val_accuracy: 0.7108 - val_loss: 0.7302
Epoch 6/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 166s 1s/step - accuracy: 0.6649 - loss: 0.8456 - val_accuracy: 0.7403 - val_loss: 0.6864
Epoch 7/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 166s 1s/step - accuracy: 0.7104 - loss: 0.7762 - val_accuracy: 0.7336 - val_loss: 0.6904
Epoch 8/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 166s 1s/step - accuracy: 0.7514 - loss: 0.6605 - val_accu

In [24]:
loss, accuracy = model.evaluate(X_test, y_test)

print("\nAccuracy :", accuracy)

65/65 ━━━━━━━━━━━━━━━━━━━━ 16s 251ms/step - accuracy: 0.9130 - loss: 0.2499

Accuracy : 0.9129593968391418


In [28]:
def detect_language(sentence):

    sentence = sentence.lower()

    seq = tokenizer.texts_to_sequences([sentence])

    seq = pad_sequences(
        seq,
        maxlen=max_len,
        padding='post'
    )

    prediction = model.predict(seq, verbose=0)

    predicted_index = np.argmax(prediction)

    confidence = np.max(prediction) * 100

    language = encoder.inverse_transform([predicted_index])

    return language[0], confidence

In [29]:
def translate_to_tamil(text):

    translated = translator.translate(
        text,
        dest='ta'
    )

    return translated.text


print("\nSupported Languages:")
print(data["Language"].unique())


Supported Languages:
['English' 'Malayalam' 'Hindi' 'Tamil' 'Portugeese' 'French' 'Dutch'
 'Spanish' 'Greek' 'Russian' 'Danish' 'Italian' 'Turkish' 'Sweedish'
 'Arabic' 'German' 'Kannada']


In [30]:
while True:

    user_text = input("\nEnter Text : ")

    if user_text.lower() == "exit":
        print("Program Ended")
        break

    # DETECT LANGUAGE
    language, confidence = detect_language(user_text)

    print("\nDetected Language :", language)

    print("Confidence:{:.2f}%".format(confidence))


    # TRANSLATE TO TAMIL
    tamil_text = translate_to_tamil(user_text)

    print("Tamil Translation :", tamil_text)


Enter Text : Je suis très heureux de vous rencontrer

Detected Language : French
Confidence:98.94%
Tamil Translation : உங்களை சந்தித்ததில் மிகவும் மகிழ்ச்சி அடைகிறேன்

Enter Text : exit
Program Ended
